In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/dataset_selection_sans_leger.csv")

print(f"Nombre total d'images : {len(df)}")
print(f"\nRépartition des labels :")
print(df['label'].value_counts())

Nombre total d'images : 6047

Répartition des labels :
label
mda         1552
glaucome    1500
diabete     1500
normaux     1495
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)

print(f"Train : {len(df_train)}")
print(f"Test  : {len(df_test)}")

classes = sorted(df['label'].unique().tolist())
print(f"Classes : {classes}")

Train : 4837
Test  : 1210
Classes : ['diabete', 'glaucome', 'mda', 'normaux']


In [3]:
def crop_black_border(img, thr=10, pad=10):
    """Enlève les bords noirs d'une image (numpy array grayscale ou BGR)."""
    gray = img if len(img.shape) == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > thr
    if not mask.any():
        return img
    ys, xs = np.where(mask)
    y0, y1 = max(0, ys.min() - pad), min(img.shape[0] - 1, ys.max() + pad)
    x0, x1 = max(0, xs.min() - pad), min(img.shape[1] - 1, xs.max() + pad)
    return img[y0:y1+1, x0:x1+1]

def load_image_gray(path):
    """Charge une image en niveaux de gris avec crop des bords noirs."""
    img = np.array(Image.open(path).convert("L"))
    img = crop_black_border(img)
    return img

def augment_image(img):
    """Retourne 3 versions augmentées: 2 rotations + 1 flip."""
    aug = []
    for angle in [-15, 15]:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w//2, h//2), angle, 1)
        aug.append(cv2.warpAffine(img, M, (w, h)))
    aug.append(cv2.flip(img, 1))
    return aug

print("Fonctions de prétraitement définies")

Fonctions de prétraitement définies


In [4]:
from skimage.feature import hog

IMG_SIZE = (128, 128)

def extract_hog(image):
    image = cv2.resize(image, IMG_SIZE)
    return hog(image, orientations=9, pixels_per_cell=(8, 8),
               cells_per_block=(2, 2), block_norm='L2-Hys')

def extract_pixels(image):
    image = cv2.resize(image, IMG_SIZE)
    return image.flatten().astype(np.float32) / 255.0

print(f"HOG: {extract_hog(np.zeros(IMG_SIZE, dtype=np.uint8)).shape[0]} features")
print(f"Pixels: {IMG_SIZE[0]*IMG_SIZE[1]} features")

HOG: 8100 features
Pixels: 16384 features


In [5]:
X_train_hog, X_train_pixels, y_train = [], [], []

for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Train"):
    img = load_image_gray(row['path'])

    # Image originale
    X_train_hog.append(extract_hog(img))
    X_train_pixels.append(extract_pixels(img))
    y_train.append(row['label'])

    # Data augmentation
    for aug in augment_image(img):
        X_train_hog.append(extract_hog(aug))
        X_train_pixels.append(extract_pixels(aug))
        y_train.append(row['label'])

X_train_hog = np.array(X_train_hog)
X_train_pixels = np.array(X_train_pixels, dtype=np.float32)
y_train = np.array(y_train)

print(f"X_train_hog:    {X_train_hog.shape}")
print(f"X_train_pixels: {X_train_pixels.shape}")
print(f"y_train:        {y_train.shape}")

Train: 100%|██████████| 4837/4837 [06:14<00:00, 12.92it/s]


X_train_hog:    (19348, 8100)
X_train_pixels: (19348, 16384)
y_train:        (19348,)


In [6]:
X_test_hog, X_test_pixels, y_test = [], [], []

for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Test"):
    img = load_image_gray(row['path'])
    X_test_hog.append(extract_hog(img))
    X_test_pixels.append(extract_pixels(img))
    y_test.append(row['label'])

X_test_hog = np.array(X_test_hog)
X_test_pixels = np.array(X_test_pixels, dtype=np.float32)
y_test = np.array(y_test)

print(f"X_test_hog:    {X_test_hog.shape}")
print(f"X_test_pixels: {X_test_pixels.shape}")

Test: 100%|██████████| 1210/1210 [01:07<00:00, 18.05it/s]

X_test_hog:    (1210, 8100)
X_test_pixels: (1210, 16384)


In [7]:
from sklearn.preprocessing import StandardScaler

# Standardiser HOG
scaler_hog = StandardScaler()
X_train_hog_sc = scaler_hog.fit_transform(X_train_hog)
X_test_hog_sc = scaler_hog.transform(X_test_hog)

# Standardiser pixels
scaler_pix = StandardScaler()
X_train_pix_sc = scaler_pix.fit_transform(X_train_pixels)
X_test_pix_sc = scaler_pix.transform(X_test_pixels)

# Combiné (pixels + HOG) pour PCA+HOG
X_train_combined = np.hstack([X_train_pix_sc, X_train_hog_sc])
X_test_combined = np.hstack([X_test_pix_sc, X_test_hog_sc])

n_pixel_features = X_train_pix_sc.shape[1]
n_hog_features = X_train_hog_sc.shape[1]

print(f"HOG scaled:     {X_train_hog_sc.shape}")
print(f"Pixels scaled:  {X_train_pix_sc.shape}")
print(f"Combined:       {X_train_combined.shape}")

HOG scaled:     (19348, 8100)
Pixels scaled:  (19348, 16384)
Combined:       (19348, 24484)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from scipy.stats import loguniform

# Distribution des paramètres (au lieu d'une grille exhaustive)
lr_dist = {
    {
        "solver": ["lbfgs"],
        "penalty": ["l2"],
        "C": [0.001, 0.01, 0.1, 1, 10],
        "max_iter": [2000, 5000]
    }
}

N_ITER = 40          # nombre de combinaisons testées par approche
pca_components = [50, 100, 200, 300]

results = {}
print(f"RandomizedSearchCV: {N_ITER} combinaisons × 3 folds = {N_ITER*3} fits par approche")

RandomizedSearchCV: 40 combinaisons × 3 folds = 120 fits par approche


In [ ]:
print("="*60)
print("1) RandomSearch: HOG seul")
print("="*60)

lr_hog = LogisticRegression(random_state=42)
search_hog = RandomizedSearchCV(
    lr_hog, lr_dist, n_iter=N_ITER, scoring="f1_macro",
    cv=3, n_jobs=2, verbose=1, random_state=42
)
search_hog.fit(X_train_hog_sc, y_train)

print(f"\nMeilleurs paramètres: {search_hog.best_params_}")
print(f"Meilleur F1 CV: {search_hog.best_score_:.4f}")

results['HOG'] = {
    'best_model': search_hog.best_estimator_,
    'best_params': search_hog.best_params_,
    'best_cv_score': search_hog.best_score_,
    'X_test': X_test_hog_sc
}

1) RandomSearch: HOG seul
Fitting 3 folds for each of 40 candidates, totalling 120 fits



Meilleurs paramètres: {'C': np.float64(0.0010656401760606447), 'class_weight': None, 'max_iter': 3000, 'penalty': 'l2', 'solver': 'lbfgs', 'tol': np.float64(0.013199942261535026)}
Meilleur F1 CV: 0.7462


: 

In [ ]:
print("="*60)
print("2) RandomSearch: PCA seul")
print("="*60)

pipe_pca = Pipeline([
    ('pca', PCA(random_state=42)),
    ('lr', LogisticRegression(random_state=42))
])

dist_pca = {'pca__n_components': pca_components}
for k, v in lr_dist.items():
    dist_pca[f'lr__{k}'] = v

search_pca = RandomizedSearchCV(
    pipe_pca, dist_pca, n_iter=N_ITER, scoring="f1_macro",
    cv=3, n_jobs=2, verbose=1, random_state=42
)
search_pca.fit(X_train_pix_sc, y_train)

print(f"\nMeilleurs paramètres: {search_pca.best_params_}")
print(f"Meilleur F1 CV: {search_pca.best_score_:.4f}")
print(f"Meilleur n_components: {search_pca.best_params_['pca__n_components']}")

results['PCA'] = {
    'best_model': search_pca.best_estimator_,
    'best_params': search_pca.best_params_,
    'best_cv_score': search_pca.best_score_,
    'X_test': X_test_pix_sc
}

2) RandomSearch: PCA seul
Fitting 3 folds for each of 40 candidates, totalling 120 fits


In [ ]:
print("="*60)
print("3) RandomSearch: PCA + HOG")
print("="*60)

pixel_cols = list(range(n_pixel_features))
hog_cols = list(range(n_pixel_features, n_pixel_features + n_hog_features))

pipe_combined = Pipeline([
    ('features', ColumnTransformer([
        ('pca', PCA(random_state=42), pixel_cols),
        ('hog', 'passthrough', hog_cols)
    ])),
    ('lr', LogisticRegression(random_state=42))
])

dist_combined = {'features__pca__n_components': pca_components}
for k, v in lr_dist.items():
    dist_combined[f'lr__{k}'] = v

search_combined = RandomizedSearchCV(
    pipe_combined, dist_combined, n_iter=N_ITER, scoring="f1_macro",
    cv=3, n_jobs=2, verbose=1, random_state=42
)
search_combined.fit(X_train_combined, y_train)

print(f"\nMeilleurs paramètres: {search_combined.best_params_}")
print(f"Meilleur F1 CV: {search_combined.best_score_:.4f}")
print(f"Meilleur n_components: {search_combined.best_params_['features__pca__n_components']}")

results['PCA + HOG'] = {
    'best_model': search_combined.best_estimator_,
    'best_params': search_combined.best_params_,
    'best_cv_score': search_combined.best_score_,
    'X_test': X_test_combined
}

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

print("="*70)
print("ÉVALUATION SUR LE JEU DE TEST")
print("="*70)

comparison_data = []

for name, data in results.items():
    model = data['best_model']
    y_pred = model.predict(data['X_test'])

    acc = accuracy_score(y_test, y_pred)
    bacc = balanced_accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    results[name]['test_accuracy'] = acc
    results[name]['test_f1'] = f1
    results[name]['y_pred'] = y_pred

    comparison_data.append({
        'Approche': name, 'CV F1': data['best_cv_score'],
        'Test Accuracy': acc, 'Balanced Acc': bacc,
        'Precision': prec, 'Recall': rec, 'F1 Score': f1
    })

    print(f"\n--- {name} ---")
    print(f"Paramètres: {data['best_params']}")
    print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")

comparison_df = pd.DataFrame(comparison_data).round(4)
print("\n" + "="*70)
print("TABLEAU COMPARATIF")
print("="*70)
print(comparison_df.to_string(index=False))

best_approach = comparison_df.loc[comparison_df['F1 Score'].idxmax(), 'Approche']
print(f"\n>>> Meilleure approche: {best_approach}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, data) in enumerate(results.items()):
    cm = confusion_matrix(y_test, data['y_pred'], labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=axes[idx], values_format='d', cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}\nF1: {data["test_f1"]:.4f}')
    axes[idx].set_xlabel('Prédit')
    axes[idx].set_ylabel('Vrai')
    plt.setp(axes[idx].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# Rapport détaillé meilleure approche
print(f"RAPPORT DÉTAILLÉ: {best_approach}")
print("="*50)
print(classification_report(y_test, results[best_approach]['y_pred'],
                            target_names=classes, zero_division=0))

# Graphique comparatif
metrics = ['CV F1', 'Test Accuracy', 'Balanced Acc', 'F1 Score']
x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, approach in enumerate(results.keys()):
    row = comparison_df[comparison_df['Approche'] == approach].iloc[0]
    values = [row[m] for m in metrics]
    bars = ax.bar(x + i*width, values, width, label=approach)
    for j, v in enumerate(values):
        ax.text(x[j] + i*width, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)

ax.set_ylabel('Score')
ax.set_title('Comparaison des 3 approches - Régression Logistique')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()